# Run 1 — Teleportation on Nighthawk (ibm_phoenix)

**What this is:** an exact repeat of your Oct 19, 2026 pilot — the same
dynamic teleportation circuit (registers `alice0` / `alice1` / `bob`,
mid-circuit measurements, feed-forward X/Z corrections) and the same
job shape: **100 shots × 6 PUBs** (one PUB per test state). The only
change is the machine: **ibm_phoenix (120-qubit Nighthawk r2)** instead
of ibm_fez (Heron r2). That gives a clean Heron-vs-Nighthawk comparison
and your first run on Nighthawk.

**Cost:** ibm_phoenix is on your **pay-as-you-go** instance
(QUANTUMCIRCUITV.001), not the free plan. The Oct 19 jobs used 2 seconds
each; expect the same ballpark here.

**Safety:** `MODE` starts as `"test"` — the whole notebook runs on a
local simulator, free, no account needed. Flip it to `"hardware"` only
when you have a working personal API key for the pay-as-you-go instance.
Your key is typed into a hidden prompt and is **never saved** in this
notebook.

In [ ]:
# Colab does not ship with Qiskit in this session -- install it first.
# This takes about a minute. Run this cell, then use Runtime > Run all.
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0
print("Packages ready: qiskit 2.6.1, qiskit-aer, qiskit-ibm-runtime.")

In [ ]:
# "test"     = local simulator (free, no account) -- run this first
# "hardware" = real ibm_phoenix on the pay-as-you-go instance
MODE = "test"

BACKEND_NAME = "ibm_phoenix"   # Nighthawk r2, 120 qubits
SHOTS = 100                    # same as the Oct 19 pilot

# Pay-as-you-go instance (QUANTUMCIRCUITV.001) -- used only in hardware mode
INSTANCE_CRN = "PASTE_YOUR_INSTANCE_CRN_HERE"
print("MODE:", MODE, "| backend:", BACKEND_NAME, "| shots:", SHOTS)

In [ ]:
if MODE == "hardware":
    import getpass
    from qiskit_ibm_runtime import QiskitRuntimeService

    token = getpass.getpass("Paste your IBM Cloud API key (hidden): ")
    service = QiskitRuntimeService(channel="ibm_quantum_platform",
                                   token=token, instance=INSTANCE_CRN)
    backend = service.backend(BACKEND_NAME)
    print("Connected:", backend.name, "-", backend.num_qubits, "qubits")
else:
    from qiskit_aer import AerSimulator
    backend = AerSimulator()
    print("TEST MODE: local simulator, no account, no cost.")

## The six test states and the teleportation circuit

In [ ]:
import numpy as np

# The six standard test states (same set as the Lab Pack notebook; the
# tomography dataset in the pack uses |-i> as the sixth state).
PREPS = {
    "|0>":  lambda c: None,
    "|1>":  lambda c: c.x(0),
    "|+>":  lambda c: c.h(0),
    "|->":  lambda c: (c.x(0), c.h(0)),
    "|+i>": lambda c: (c.h(0), c.s(0)),
    "|-i>": lambda c: (c.h(0), c.sdg(0)),
}
STATES = list(PREPS.keys())

# Theory: Bob's Z-basis probability of measuring 0 for each input state.
THEORY_P0 = {"|0>": 1.0, "|1>": 0.0, "|+>": 1.0, "|->": 0.0,
             "|+i>": 1.0, "|-i>": 0.0}  # matching-basis readout (fixed 2026-10-28)
print("States:", STATES)

In [ ]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister


def teleportation_circuit(prep, basis="Z"):
    """Full dynamic-circuit teleportation, matching the Oct 19, 2026
    ibm_fez job layout: registers alice0 / alice1 / bob, mid-circuit
    measurements, and classically-controlled X and Z corrections.

    basis: Bob's readout basis ("Z" = direct, "X"/"Y" rotate first)."""
    qr = QuantumRegister(3, "q")
    alice0 = ClassicalRegister(1, "alice0")
    alice1 = ClassicalRegister(1, "alice1")
    bob = ClassicalRegister(1, "bob")
    qc = QuantumCircuit(qr, alice0, alice1, bob)

    prep(qc)                       # 1) prepare the state to teleport (q0)
    qc.barrier()
    qc.h(1); qc.cx(1, 2)           # 2) Bell pair q1-q2
    qc.barrier()
    qc.cx(0, 1); qc.h(0)           # 3) Alice's Bell-basis rotation
    qc.measure(0, alice0[0])
    qc.measure(1, alice1[0])
    qc.barrier()
    with qc.if_test((alice1[0], 1)):   # 4) feed-forward corrections
        qc.x(2)
    with qc.if_test((alice0[0], 1)):
        qc.z(2)
    qc.barrier()
    if basis == "X":               # 5) Bob's readout in the chosen basis
        qc.h(2)
    elif basis == "Y":
        qc.sdg(2); qc.h(2)
    qc.measure(2, bob[0])
    return qc


BASES = {"|0>": "Z", "|1>": "Z", "|+>": "X", "|->": "X", "|+i>": "Y", "|-i>": "Y"}
circuits = [teleportation_circuit(PREPS[s], basis=BASES[s]) for s in STATES]  # matching-basis readout (fixed 2026-10-28)
print(f"Built {len(circuits)} teleportation circuits (one per state).")
circuits[0].draw("mpl")

## Run it

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuits = pm.run(circuits)

if MODE == "hardware":
    from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler
    sampler = RuntimeSampler(mode=backend)
else:
    from qiskit.primitives import BackendSamplerV2
    sampler = BackendSamplerV2(backend=backend)

pubs = [(isa,) for isa in isa_circuits]
job_or_result = sampler.run(pubs, shots=SHOTS)
result = job_or_result.result() if hasattr(job_or_result, "result") else job_or_result
if MODE == "hardware":
    print("Job ID:", job_or_result.job_id())
print(f"Got results for {len(result)} PUBs at {SHOTS} shots each.")

## Read the results

For each state: Bob's measured P(0) vs theory, and the deviation.
Computational states (|0>, |1>) should land near 100% / 0%; the four
superposition states should land near 50%. On the simulator expect
sampling noise of ~5 points at 100 shots; on hardware expect more —
that extra spread *is* the Heron-vs-Nighthawk story.

In [ ]:
import json

rows = []
print(f"{'State':<7}{'Bob P(0)':>9}{'Theory':>8}{'|Diff|':>8}")
for state, pub_result in zip(STATES, result):
    counts = pub_result.data.bob.get_counts()
    p0 = counts.get("0", 0) / SHOTS
    theory = THEORY_P0[state]
    rows.append({"state": state, "counts": {k: int(v) for k, v in counts.items()},
                 "p0_measured": p0, "p0_theory": theory})
    print(f"{state:<7}{p0:>9.3f}{theory:>8.3f}{abs(p0 - theory):>8.3f}")

mean_dev = float(np.mean([abs(r["p0_measured"] - r["p0_theory"]) for r in rows]))
print(f"\nMean |deviation| from theory: {mean_dev:.3f}")
print("(Oct 19 ibm_fez pilot for comparison: one job's PUB came back ~45/55.)")

with open("phoenix_teleportation_results.json", "w") as f:
    json.dump({"backend": BACKEND_NAME if MODE == "hardware" else "AerSimulator",
               "shots": SHOTS, "rows": rows}, f, indent=2)
print("Saved phoenix_teleportation_results.json")

## When this finishes on hardware

Send me the saved JSON (or a screenshot of the table). Next steps:
compare state-by-state against the Oct 19 ibm_fez numbers, then the
tomography run (Run 2 notebook) completes the hardware dataset for the
Hardware Edition of the Lab Pack.